# Kapitel 1 — Daten lesen · Lösung

*Mathe für KI, Schritt für Schritt*, Belk Siri — Begleit-Notebook.

[In Google Colab öffnen](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/de/ch01_corrige.ipynb) · Begleitseite: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/de.html

Jede Aufgabe greift die Aufgabenstellung des Buchs auf. Die Antwort beginnt mit der Überlegung auf Papier und zeigt dann ihre Entsprechung in Python.

**Konvention.** Wie im Buch teilen Varianz, Standardabweichung, z-Wert und Kovarianz durch $n$ (`ddof=0`), sofern nicht ausdrücklich anders angegeben.

**Inhalt.** Daten · Die Berechnungen der Lektionen · Aufgaben 1.1 bis 1.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (7, 2.6)
# Farben des Buchs: eine Farbe = eine Rolle (beobachtet, vorhergesagt, Abweichung, Gerade des Kollegen)
BLUE, ORANGE, RESIDUAL, GREEN = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Daten

Die Kennungen (D0.1, D1.1…) sind die des Buchs und der CSV-Dateien.

In [ ]:
# Daten von Kapitel 1, aus den CSV-Dateien des Buchs übernommen (nicht ändern)

# D0.1 — die sechs Wohnungen der Einleitung
homes = pd.DataFrame({
    'home':   ['A', 'B', 'C', 'D', 'E', 'F'],
    'floor_area': [32, 45, 54, 70, 85, 98],
    'price':  [118, 172, 185, 245, 279, 335],
})

# D1.1 — Verkaufspreise der T2-Wohnungen (zwei Zimmer, k€), Lektion 1.1
north = [174, 177, 181, 183, 185]
south  = [150, 165, 185, 195, 205]

# D1.2 — Verkaufsregister des Viertels Süd, Lektionen 1.2 bis 1.5
sales_log = pd.DataFrame({
    'sale':       ['S01', 'S02', 'S03', 'S04', 'S05', 'S06', 'S07', 'S08', 'S09', 'S10', 'S11', 'S12', 'S13'],
    'rooms':      [1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 4, 4, 4],
    'floor_area':  [24, 27, 43, 40, 44, 46, 52, 61, 64, 70, 82, 88, 95],
    'energy_class':         ['D', 'C', 'F', 'D', 'C', 'D', 'C', 'E', 'D', 'B', 'E', 'C', 'G'],
    'price':   [96, 108, 150, 165, 185, 195, 205, 228, 241, 276, 289, 318, 260],
    'days_to_sell': [9, 18, 97, 21, 26, 23, 31, 14, 35, 28, 42, 58, 183],
})

# D1.5 — Preis pro m² nach Stockwerk (k€/m²), Lektion 1.5
floors = pd.DataFrame({
    'floor':        [0, 1, 2, 3, 4, 5, 6],
    'price_per_m2': [3.0, 3.3, 3.5, 3.6, 3.5, 3.3, 3.0],
})

# D1.E1 — Verkaufsdauern von drei Agenturen (Tage), Aufgabe 1.1
agency_delays = {'P': [28, 29, 30, 31, 32], 'Q': [20, 25, 30, 35, 40], 'R': [10, 28, 30, 32, 50]}

# D1.E2 — Monatsmieten der Studios (€), Aufgabe 1.2
tilleuls = [585, 590, 600, 605, 605, 615]
port     = [520, 560, 590, 620, 640, 670]

### Die Variablen der Daten

Die Namen im Code sind englisch. Für die Spalten gibt die letzte Spalte den Namen an, der in den CSV-Dateien des Buchs verwendet wird.

| Name | Bedeutung | Einheit | Spalte der CSV-Datei |
|:--|:--|:--|:--|
| `home` | Kennung der Wohnung (A, B…) | — | `logement` |
| `floor_area` | Wohnfläche der Wohnung | m² | `surface_m2` |
| `price` | Verkaufspreis | k€ | `prix_kEUR` |
| `sale` | Kennung des Verkaufs (S01…) | — | `vente` |
| `rooms` | Anzahl der Zimmer | Zimmer | `pieces` |
| `energy_class` | Energieeffizienzklasse (A bis G) | — | `dpe` |
| `days_to_sell` | Verkaufsdauer | Tage | `delai_jours` |
| `floor` | Stockwerk | — | `etage` |
| `price_per_m2` | Preis pro Quadratmeter | k€/m² | `prix_m2_kEUR` |

In [ ]:
# Bereitgestellte Werkzeuge: Sie müssen sie nicht ändern

def book_quartiles(values):
    """Q1 und Q3 nach der Regel des Buchs (Gleichung 1.5): Ränge ⌈n/4⌉ und ⌈3n/4⌉."""
    x = sorted(values)
    n = len(x)
    rank_q1 = -(-n // 4)          # n/4 auf die nächste ganze Zahl aufgerundet
    rank_q3 = -(-3 * n // 4)      # 3n/4 auf die nächste ganze Zahl aufgerundet
    return x[rank_q1 - 1], x[rank_q3 - 1]   # Rang 1 = Position 0 in Python


def dot_rows(series, xlabel, bands=None, ax=None):
    """Eine Punktreihe pro Datenreihe, wie in den Abbildungen von Lektion 1.1.
    bands: Wörterbuch {Name: (Mittelwert, Standardabweichung)}, um Mittelwert ± Standardabweichung zu zeichnen."""
    if ax is None:
        fig, ax = plt.subplots()
    names = list(series)
    colors = [BLUE, ORANGE, '#52514e']
    for i, name in enumerate(names):
        y = len(names) - 1 - i
        if bands and name in bands:
            m, s = bands[name]
            ax.add_patch(plt.Rectangle((m - s, y - 0.25), 2 * s, 0.5, color='#e9e8e4'))
            ax.plot([m, m], [y - 0.3, y + 0.3], color='black')
        ax.scatter(series[name], [y] * len(series[name]), s=50,
                   color=colors[i % 3], zorder=3)
    ax.set_yticks(range(len(names)), list(reversed(names)))
    ax.set_ylim(-0.6, len(names) - 0.4)
    ax.set_xlabel(xlabel)
    ax.grid(axis='x', alpha=0.4)
    return ax

## Die Berechnungen der Lektionen, in Python

Diese Zellen wiederholen die Berechnungen der Lektionen 1.1 bis 1.5. Führen Sie sie aus und vergleichen Sie mit dem Buch. Hier werden `np.std` und `np.var` mit `ddof=0` aufgerufen, also mit einer Division durch $n$, wie im Kapitel.

In [ ]:
# Lektion 1.1 — Mittelwert, Varianz und Standardabweichung (Division durch n)
for name, one_series in [('Nord', north), ('Süd', south)]:
    print(f'{name}: Mittelwert = {np.mean(one_series):g} k€, Varianz = {np.var(one_series, ddof=0):g} k€², '
          f'Standardabweichung = {np.std(one_series, ddof=0):g} k€')

In [ ]:
# Lektion 1.2 — Häufigkeiten je Energieeffizienzklasse (ordinale qualitative Variable)
sales_log['energy_class'].value_counts().reindex(list('ABCDEFG'), fill_value=0)

In [ ]:
# Lektion 1.3 — Mittelwert, Median und Quartile der Verkaufsdauern
d = sales_log['days_to_sell']
print('Mittelwert:', d.mean(), ' Median:', d.median())
print('ohne S13: Mittelwert', d[sales_log['sale'] != 'S13'].mean(),
      ' Median', d[sales_log['sale'] != 'S13'].median())
print('Quartile, Regel des Buchs (Gleichung 1.5):', book_quartiles(d))
print('Quartile, NumPy-Standard (Interpolation):', np.quantile(d, [0.25, 0.75]))

In [ ]:
# Lektion 1.4 — standardisierte Werte (Gleichung 1.6, Standardabweichung mit Division durch n)
for name, one_series, sale in [('Nord', north, 188), ('Süd', south, 210)]:
    m, s = np.mean(one_series), np.std(one_series, ddof=0)
    print(f'{name}: z der Reihe = {np.round((np.array(one_series) - m) / s, 4)}, '
          f'z des Verkaufs zu {sale} k€ = {(sale - m) / s:g}')

In [ ]:
# Lektion 1.5 — Kovarianz und Korrelation der fünf T2-Wohnungen im Süden (Division durch n)
t2 = sales_log[sales_log['rooms'] == 2]
x, y = t2['floor_area'].to_numpy(float), t2['price'].to_numpy(float)
cov = np.mean((x - x.mean()) * (y - y.mean()))
print('Kovarianz:', cov, 'm²·k€   r:', cov / (x.std(ddof=0) * y.std(ddof=0)))
print('r mit np.corrcoef:', np.corrcoef(x, y)[0, 1].round(4))
print('r Stockwerk / Preis pro m²:', np.corrcoef(floors['floor'], floors['price_per_m2'])[0, 1].round(4))

## Aufgabe 1.1 — Ein Diagramm lesen

*Daten D1.E1. Begriffe: Mittelwert und Standardabweichung (Lektion 1.1).* Das folgende Diagramm gibt das aus Aufgabe 1.1 des Buchs wieder.

In [ ]:
dot_rows({f'Agentur {a}': v for a, v in agency_delays.items()},
                  'Verkaufsdauer (Tage)')
plt.xlim(0, 60)
plt.show()

**a) und b)** Schätzen Sie nach Augenmaß die mittlere Verkaufsdauer jeder Agentur und ordnen Sie die Agenturen dann von der am wenigsten zur am stärksten gestreuten.

In [ ]:
estimated_mean = {'P': 30, 'Q': 30, 'R': 30}   # in Tagen
spread_order = ['P', 'Q', 'R']

In [ ]:
# Prüfung (Division durch n)
for a, v in agency_delays.items():
    print(f'Agentur {a}: Mittelwert = {np.mean(v):g} T, Standardabweichung = {np.std(v, ddof=0):.2f} T')
true_order = sorted(agency_delays, key=lambda a: np.std(agency_delays[a]))
if spread_order is None or None in estimated_mean.values():
    print('Bitte füllen Sie die vorherige Zelle aus.')
else:
    print('Reihenfolge der Streuung:', 'richtig' if spread_order == true_order else 'bitte prüfen')

**c)** Zählen Sie für Q und für R die Verkäufe, die höchstens 2 Tage vom Mittelwert entfernt abgeschlossen wurden.

In [ ]:
close_to_Q = 1
close_to_R = 3

In [ ]:
# Prüfung
for a, yours in [('Q', close_to_Q), ('R', close_to_R)]:
    v = np.array(agency_delays[a])
    truth = int(np.sum(np.abs(v - v.mean()) <= 2))
    if yours is None:
        print(f'Agentur {a}: noch nicht ausgefüllt.')
    else:
        print(f"Agentur {a}: {'richtig' if yours == truth else 'bitte prüfen'}")

### Lösung zu Aufgabe 1.1

**a)** Die drei Reihen sind symmetrisch um 30 Tage. Die mittlere Verkaufsdauer beträgt für jede Agentur **30 Tage**, wie die Rechnung bestätigt: (28 + 29 + 30 + 31 + 32) / 5 = 150 / 5 = 30, ebenso für Q und R.

**b)** Von der am wenigsten zur am stärksten gestreuten: **P, Q, R**. Die Punkte von P liegen zwischen 28 und 32 Tagen. Die von Q verteilen sich gleichmäßig von 20 bis 40 Tagen. Die von R reichen von 10 bis 50 Tagen. Standardabweichungen (Division durch $n$): P ≈ 1,41 T; Q ≈ 7,07 T; R ≈ 12,71 T.

**c)** Q: **1** Verkauf (30 Tage). R: **3** Verkäufe (28, 30 und 32 Tage). Dennoch hat R die größte Standardabweichung. Seine beiden extremen Verkäufe, nach 10 und 50 Tagen, liegen 20 Tage vom Mittelwert entfernt. Ihre Quadrate betragen je 400: Die Quadratsumme von R beträgt 400 + 4 + 0 + 4 + 400 = 808, gegenüber 100 + 25 + 0 + 25 + 100 = 250 für Q. Die Standardabweichung fasst alle Abweichungen zusammen, und das Quadrieren gibt den größten ein hohes Gewicht. Drei Verkäufe nahe der Mitte gleichen zwei sehr weit entfernte Verkäufe nicht aus.

*Erwartete Elemente:* die Reihenfolge P, Q, R; die beiden Zählungen; die Rolle der beiden Extremwerte und der Quadrate in der Rechnung.

## Aufgabe 1.2 — Zwei Wohnanlagen, dieselbe mittlere Miete

*Daten D1.E2. Begriffe: Mittelwert, Varianz und Standardabweichung (Lektion 1.1).* Camille kann höchstens 570 € im Monat ausgeben. Die Fragen a) bis d) lösen Sie zuerst auf Papier, mit der Tabelle aus Aufgabe 1.2 des Buchs. Dieses Notebook dient dazu, Ihre Antworten zu prüfen, dann die Standardabweichung zu programmieren (Frage e) und zwei Konventionen zu vergleichen (Frage f).

In [ ]:
dot_rows({'Les Tilleuls': tilleuls, 'Le Port': port}, "Monatsmiete eines Studios (€)")
plt.axvline(570, color='black', linestyle=':', label='Budget von Camille: 570 €')
plt.xlim(500, 700)
plt.legend(loc='upper right', fontsize=8)
plt.show()

### a) Die Mieten ablesen

Wie viele Studios kann Camille in jeder Wohnanlage mit 570 € mieten, und wie viele mit 620 €?

In [ ]:
affordable = {
    570: {'Les Tilleuls': 0, 'Le Port': 2},
    620: {'Les Tilleuls': 6, 'Le Port': 4},
}

In [ ]:
# Prüfung: `<=` liefert Wahr/Falsch-Werte, np.sum zählt die „Wahr“-Werte
for budget, answers in affordable.items():
    for name, one_series in [('Les Tilleuls', tilleuls), ('Le Port', port)]:
        truth = int(np.sum(np.array(one_series) <= budget))
        yours = answers[name]
        status = 'noch nicht ausgefüllt' if yours is None else ('richtig' if yours == truth else 'bitte prüfen')
        print(f'Budget {budget} €, {name}: {status}')

### Lösung zu 1.2 a)

**Beim Lesen der Liste.** Mit 570 € findet Camille in Les Tilleuls **kein** Studio: Das günstigste kostet 585 €. In Le Port sind **zwei** von sechs Studios für sie erreichbar, zu 520 und 560 €. Die Wohnanlage, in der sie eine Chance hat, ist also die mit den am stärksten gestreuten Mieten. Mit 620 € kehrt sich die Lage um: Alle **sechs** Studios in Les Tilleuls sind für sie erreichbar, gegenüber **vier** von sechs in Le Port (520, 560, 590 und 620 €). Die beiden gleichen Mittelwerte ließen von diesen Unterschieden nichts ahnen.

### b) Die beiden Mittelwerte

**Auf Papier.** Les Tilleuls: 585 + 590 + 600 + 605 + 605 + 615 = 3600, und 3600 / 6 = **600 €**. Le Port: 520 + 560 + 590 + 620 + 640 + 670 = 3600, und 3600 / 6 = **600 €**.

**In Python.** `np.mean` bildet dieselbe Summe und dann dieselbe Division.

In [ ]:
mean_tilleuls_by_hand = 600
mean_port_by_hand = 600

In [ ]:
# Prüfung
for name, one_series, yours in [('Les Tilleuls', tilleuls, mean_tilleuls_by_hand),
                              ('Le Port', port, mean_port_by_hand)]:
    if yours is None:
        print(f'{name}: bitte füllen Sie die vorherige Zelle aus.')
    else:
        ok = np.isclose(yours, np.mean(one_series))
        print(f"{name}: {'richtig' if ok else 'bitte prüfen'} (Ihr Wert: {yours} €)")

### c) Varianz und Standardabweichung von Hand (Division durch $n = 6$)

**Les Tilleuls** ($\bar{x} = 600$ €)

| $x_i$ (€) | 585 | 590 | 600 | 605 | 605 | 615 | Summe |
|:--|--:|--:|--:|--:|--:|--:|--:|
| $x_i - \bar{x}$ (€) | −15 | −10 | 0 | +5 | +5 | +15 | 0 |
| $(x_i - \bar{x})^2$ (€²) | 225 | 100 | 0 | 25 | 25 | 225 | 600 |

Varianz: 600 / 6 = **100 €²**. Standardabweichung: √100 = **10 €**.

**Le Port** ($\bar{x} = 600$ €)

| $x_i$ (€) | 520 | 560 | 590 | 620 | 640 | 670 | Summe |
|:--|--:|--:|--:|--:|--:|--:|--:|
| $x_i - \bar{x}$ (€) | −80 | −40 | −10 | +20 | +40 | +70 | 0 |
| $(x_i - \bar{x})^2$ (€²) | 6400 | 1600 | 100 | 400 | 1600 | 4900 | 15 000 |

Varianz: 15 000 / 6 = **2500 €²**. Standardabweichung: √2500 = **50 €**.

*Zwei Kontrollen unterwegs.* In jeder Tabelle muss die Zeile der Abweichungen zusammen 0 ergeben (Lektion 1.1). Außerdem muss die Standardabweichung in derselben Größenordnung bleiben wie die Abweichungen der Tabelle: 10 € bei Abweichungen von 0 bis 15 €, 50 € bei Abweichungen von 10 bis 80 €.

**In Python.** `np.var(..., ddof=0)` und `np.std(..., ddof=0)` teilen durch $n$.

In [ ]:
variance_tilleuls_by_hand = 100   # in €²
variance_port_by_hand = 2500      # in €²
std_tilleuls_by_hand = 10  # in €
std_port_by_hand = 50      # in €

In [ ]:
# Prüfung (Division durch n: ddof=0)
your_values = {
    'Varianz Tilleuls': (variance_tilleuls_by_hand, np.var(tilleuls, ddof=0)),
    'Varianz Port': (variance_port_by_hand, np.var(port, ddof=0)),
    'Standardabweichung Tilleuls': (std_tilleuls_by_hand, np.std(tilleuls, ddof=0)),
    'Standardabweichung Port': (std_port_by_hand, np.std(port, ddof=0)),
}
for name, (yours, expected) in your_values.items():
    if yours is None:
        print(f'{name}: noch nicht ausgefüllt.')
    else:
        print(f"{name}: {'richtig' if np.isclose(yours, expected) else 'bitte prüfen'}")

### d) Interpretieren

*Beispielantwort.* In Les Tilleuls weichen die Mieten gewöhnlich um etwa 10 € von 600 € ab: Sie sind fast alle gleich. In Le Port beträgt die übliche Abweichung etwa 50 €: Dort gibt es Studios, die deutlich günstiger, und solche, die deutlich teurer als der Mittelwert sind. Die Standardabweichung reicht nicht aus, um Frage a) zu beantworten. Sie beschreibt die Streuung im Allgemeinen, sagt aber nicht, ob ein bestimmtes Studio 570 € oder weniger kostet. Eine Standardabweichung von 50 € um 600 € macht eine Miete unter 570 € plausibel. Nur die Liste erlaubt es zu zählen, dass es genau zwei sind, zu 520 und 560 €.

*Erwartete Elemente:* ein Satz pro Wohnanlage, mit Einheit; die Unterscheidung zwischen einem Streuungsmaß und der Frage „Passt ein Studio ins Budget?“; der Rückgriff auf die Mietliste.

### e) Die Standardabweichung programmieren

Vervollständigen Sie die Funktion nach Gleichung 1.3 des Buchs, mit einer Division durch $n$:

$$\sigma = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2}$$

Verwenden Sie weder `np.std` noch `np.var`: Ziel ist, jeden Schritt selbst zu schreiben.

In [ ]:
# Aufgabe 1.2 e)
def std_dev(values):
    """Standardabweichung einer Reihe, mit Division durch n (Gleichung 1.3)."""
    x = np.array(values, dtype=float)
    n = len(x)
    mean = np.sum(x) / n               # Gleichung 1.1
    deviations = x - mean                  # eine Subtraktion pro Wert
    variance = np.sum(deviations ** 2) / n    # Gleichung 1.2
    return np.sqrt(variance)              # zurück zu Euro

Jede Zeile der Funktion entspricht einer Spalte der Tabelle aus Frage c): die Abweichungen, ihre Quadrate, der Mittelwert dieser Quadrate, dann die Quadratwurzel, um zu Euro zurückzukehren. Häufige Fehler: die Wurzel vor dem Mittelwert ziehen; `np.sum(deviations) / n` berechnen, was immer 0 ergibt (Lektion 1.1).

In [ ]:
# Prüfung: Führen Sie diese Zelle aus, ohne sie zu ändern
if std_dev(tilleuls) is None:
    print('Die Funktion gibt noch None zurück: Bitte vervollständigen Sie sie.')
else:
    for name, one_series, expected in [('Les Tilleuls', tilleuls, 10.0), ('Le Port', port, 50.0)]:
        obtained = std_dev(one_series)
        status = 'richtig' if np.isclose(obtained, expected) else f'bitte prüfen (erwartet: {expected})'
        print(f'{name}: {obtained:.2f} € → {status}')

### f) Vertiefung: durch $n$ oder durch $n - 1$ teilen

Standardmäßig teilt die pandas-Methode `.std()` die Quadratsumme durch $n - 1$ (`ddof=1`). NumPy teilt standardmäßig durch $n$ (`ddof=0`), die Konvention des Kapitels.

**Auf Papier, mit Division durch $n - 1 = 5$.** Tilleuls: 600 / 5 = 120, dann √120 ≈ **10,95 €**. Port: 15 000 / 5 = 3000, dann √3000 ≈ **54,77 €**.

Beide Konventionen sind richtig, beantworten aber nicht dieselbe Frage. Die Division durch $n$ beschreibt die sechs Studios selbst. Die Division durch $n - 1$ dient dazu, wenn diese sechs Studios eine größere Gesamtheit vertreten. Kapitel 6 wird diese Wahl erklären. Mit `.std(ddof=0)` liefert pandas wieder 10 und 50 €. Geben Sie in einer Rechnung immer die verwendete Konvention an.

In [ ]:
rents = pd.DataFrame({'Les Tilleuls': tilleuls, 'Le Port': port})
print('pandas, Standard (Division durch n − 1):', rents.std().round(2).to_dict())
print('pandas, ddof=0 (Division durch n)      :', rents.std(ddof=0).round(2).to_dict())

## Aufgabe 1.3 — Ein überraschender Verkauf?

*Daten D1.1. Begriffe: standardisierter Wert (Lektion 1.4, Gleichung 1.6).* Eine T2-Wohnung (zwei Zimmer) wurde gerade für 196 k€ verkauft. Mittelwerte und Standardabweichungen: die aus Lektion 1.1, mit Division durch $n$.

In [ ]:
z_north = (196 - 180) / 4     # = 4
z_south = (196 - 180) / 20     # = 0,8
south_price_as_unusual = 180 + 4 * 20   # = 260 k€

In [ ]:
# Prüfung (Standardabweichung mit Division durch n)
expected_values = {
    'z Nord': (z_north, (196 - np.mean(north)) / np.std(north, ddof=0)),
    'z Süd': (z_south, (196 - np.mean(south)) / np.std(south, ddof=0)),
    'gleichwertiger Preis Süd': (south_price_as_unusual,
                            np.mean(south) + (196 - np.mean(north)) / np.std(north, ddof=0) * np.std(south, ddof=0)),
}
for name, (yours, truth) in expected_values.items():
    status = 'noch nicht ausgefüllt' if yours is None else ('richtig' if np.isclose(yours, truth) else 'bitte prüfen')
    print(f'{name}: {status}')

### Lösung zu Aufgabe 1.3

**a)** Im Norden: $z = (196 - 180)/4 = 16/4 =$ **4**. Im Süden: $z = (196 - 180)/20 = 16/20 =$ **0,8**. Beide Werte haben keine Einheit: Sie zählen Standardabweichungen.

**b)** Im **Norden** wäre er überraschend. In der Abbildung der standardisierten Werte in Lektion 1.4 liegt ein Wert von 4 weit jenseits aller Verkäufe im Norden, von denen keiner 1,25 übersteigt. Im Süden liegt 0,8 zwischen den Verkäufen bei 0,75 und 1,25: ein gewöhnlicher Verkauf.

**c)** Gesucht ist der Preis im Süden, dessen Wert 4 beträgt: $180 + 4 \times 20 =$ **260 k€**. Man geht von Gleichung 1.6 aus und löst nach $x$ auf: $x = \bar{x} + z\,\sigma$.

*Erwartete Elemente:* die beiden Werte; der Vergleich in Standardabweichungen statt in k€; der gleichwertige Preis von 260 k€.

## Aufgabe 1.4 — Gehören Fläche und Preis zusammen?

*Daten D0.1 (die sechs Wohnungen der Einleitung). Begriffe: Kovarianz und Korrelation (Lektion 1.5, Gleichungen 1.7 und 1.8).*

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.scatter(homes['floor_area'], homes['price'], color=BLUE)
for _, l in homes.iterrows():
    plt.annotate(l['home'], (l['floor_area'], l['price']), xytext=(-10, 4), textcoords='offset points')
plt.xlabel('Wohnfläche (m²)')
plt.ylabel('Verkaufspreis (k€)')
plt.grid(alpha=0.4)
plt.show()

**a)** Die Punkte steigen von links nach rechts und liegen fast auf einer Geraden: $r$ wird **positiv und sehr nahe bei 1** sein.

**b)** Vervollständigen Sie die folgenden Zeilen. Teilen Sie überall durch $n$.

In [ ]:
x = homes['floor_area'].to_numpy(dtype=float)
y = homes['price'].to_numpy(dtype=float)
n = len(x)

products = (x - x.mean()) * (y - y.mean())          # ein Produkt pro Wohnung
covariance = np.sum(products) / n                   # Gleichung 1.7
r = covariance / (x.std(ddof=0) * y.std(ddof=0))    # Gleichung 1.8

**Auf Papier** beträgt die mittlere Fläche 384 / 6 = 64 m² und der mittlere Preis 1334 / 6 ≈ 222,33 k€. Die Summe der sechs Produkte beträgt 9825, also eine Kovarianz von 9825 / 6 = **1637,5 m²·k€**. Die Standardabweichungen betragen etwa 22,80 m² und 72,15 k€, also $r ≈ 1637{,}5 / (22{,}80 × 72{,}15) ≈$ **0,996**.

In [ ]:
# Prüfung
if r is None or covariance is None:
    print('Bitte füllen Sie die vorherige Zelle aus.')
else:
    print('Kovarianz:', round(covariance, 2), 'm²·k€ →',
          'richtig' if np.isclose(covariance, 1637.5) else 'bitte prüfen (erwartet: 1637,5)')
    print('r:', round(r, 4), '→', 'richtig' if np.isclose(r, np.corrcoef(x, y)[0, 1]) else 'bitte prüfen')

**Eine Falle, die man kennen sollte.** `np.cov(x, y)` teilt standardmäßig durch $n - 1$ und liefert 1965, nicht 1637,5. Für die Konvention des Kapitels müssen Sie `np.cov(x, y, bias=True)` schreiben. Der Koeffizient $r$ hängt dagegen nicht von der Konvention ab, weil derselbe Faktor oben und unten auftritt (Lektion 1.5).

**c)** Es gibt keinen Widerspruch. Erstens misst $r$, wie eng sich die Punkte um *die* Gerade scharen, die am besten zu ihnen passt (die Kapitel 3 Ihnen zu berechnen beibringen wird), nicht um unsere nach Augenmaß gezeichnete Gerade. Zweitens ist Nähe relativ: Die Preise streuen mit einer Standardabweichung von etwa 72 k€, und Abweichungen von 5 bis 11 k€ im Betrag sind klein gegenüber dieser Streuung. Ein $r$ nahe 1 garantiert also keine kleinen Fehler in k€. Um die Größe der Fehler zu kennen, muss man sie berechnen, wie in Aufgabe 1.6.

*Erwartete Elemente:* „$r$ bezieht sich auf die beste Gerade“; „die Abweichungen sind klein im Verhältnis zur Streuung der Preise“; die Unterscheidung zwischen Ausrichtung und Größe der Fehler in k€.

In [ ]:
print('np.cov standardmäßig (Division durch n − 1):', np.cov(x, y)[0, 1])
print('np.cov mit bias=True (Division durch n)     :', np.cov(x, y, bias=True)[0, 1])

## Aufgabe 1.5 — Vorhersagen, dann prüfen

*Daten D1.1. Begriffe: Mittelwert und Standardabweichung (Lektion 1.1), Median und Interquartilsabstand (Lektion 1.3).* Eine sechste T2-Wohnung wird im Norden für 240 k€ verkauft.

**a)** Erwartete Vorhersage: Der Mittelwert steigt deutlich, weil er die 240 k€ auf alle Verkäufe verteilt. Der Median bewegt sich kaum, weil sich der neue Verkauf nur „hinter der Mitte“ einreiht. Die Standardabweichung steigt stark, weil eine quadrierte Abweichung von etwa 50 k€ die Summe dominiert. Der Interquartilsabstand ändert sich wenig, weil er nur von den mittleren Werten abhängt.

In [ ]:
north_plus = north + [240]
print('Nord:', sorted(north))
print('Nord mit dem Verkauf zu 240 k€:', sorted(north_plus))

**b)** Vervollständigen Sie die Funktion `summary`. Sie soll die vier Kennzahlen zurückgeben, mit der Standardabweichung durch $n$ geteilt und den Quartilen nach der Regel des Buchs berechnet (`book_quartiles`, weiter oben bereitgestellt). Die Schlüssel des Wörterbuchs sind englisch: `mean` = Mittelwert, `median` = Median, `std_dev` = Standardabweichung, `iqr` = Interquartilsabstand.

In [ ]:
def summary(one_series):
    q1, q3 = book_quartiles(one_series)
    return {
        'mean': np.mean(one_series),
        'median': np.median(one_series),
        'std_dev': np.std(one_series, ddof=0),   # Division durch n
        'iqr': q3 - q1,
    }

In [ ]:
table = pd.DataFrame({'Nord (5 Verk.)': summary(north), 'Nord + 240 k€ (6 Verk.)': summary(north_plus),
                        'Süd (5 Verk.)': summary(south)})
if table.isna().any().any():
    print('Bitte vervollständigen Sie die Funktion summary.')
else:
    expected = {'mean': (180, 190), 'median': (181, 182), 'std_dev': (4, 22.66),
               'iqr': (6, 8)}
    for k, (before, after) in expected.items():
        ok = np.isclose(table.loc[k].iloc[0], before) and np.isclose(table.loc[k].iloc[1], after, atol=0.005)
        print(f"{k}: {'richtig' if ok else 'bitte prüfen'}")
table.round(2)

In [ ]:
# Die beiden Reihen nach dem Vorbild der Abbildung „Mittelwert ± Standardabweichung“ aus Lektion 1.1
if table.isna().any().any():
    print('Bitte vervollständigen Sie die Funktion summary, um das Diagramm anzuzeigen.')
else:
    dot_rows({'Nord': north, 'Nord + 240': north_plus},
                      'Verkaufspreis einer T2-Wohnung (k€)',
                      bands={'Nord': (np.mean(north), np.std(north, ddof=0)),
                              'Nord + 240': (np.mean(north_plus), np.std(north_plus, ddof=0))})
    plt.xlim(140, 250)
    plt.show()

### Lösung zu Aufgabe 1.5

**b) Auf Papier.** Mit dem Verkauf zu 240 k€ steigt die Summe von 900 auf 1140 k€ für 6 Verkäufe: Mittelwert **190 k€** (statt 180). Sortiert lauten die sechs Werte 174, 177, 181, 183, 185, 240. $n$ ist gerade, also beträgt der Median (181 + 183)/2 = **182 k€** (statt 181). Die Abweichungen von 190 sind −16, −13, −9, −7, −5, +50. Ihre Quadrate betragen 256, 169, 81, 49, 25 und 2500, mit einer Summe von 3080. Die Varianz beträgt 3080/6 ≈ 513,3 k€² und die Standardabweichung **≈ 22,66 k€** (statt 4). Quartile (Gleichung 1.5): vorher Ränge ⌈5/4⌉ = 2 und ⌈15/4⌉ = 4, also Q1 = 177 und Q3 = 183, daher ein Interquartilsabstand von **6 k€**. Nachher Ränge ⌈6/4⌉ = 2 und ⌈18/4⌉ = 5, also Q1 = 177 und Q3 = 185, daher **8 k€**.

**c)** Nein. Fünf von sechs Verkäufen liegen weiterhin höchstens 6 k€ von 180 k€ entfernt. Ein einziger außergewöhnlicher Verkauf genügt, um die Standardabweichung von 4 auf 22,66 k€ zu heben, über die 20 k€ des Südens, weil seine Abweichung von 50 k€ mit 2500 in eine Summe von 3080 eingeht. Die robusten Maße erzählen etwas anderes: Der Interquartilsabstand im Norden steigt von 6 auf 8 k€ und bleibt weit unter dem des Südens (30 k€). Für eine gewöhnliche T2-Wohnung im Norden bleiben die Preise sehr gut vorhersehbar. Die Standardabweichung zeigt vor allem einen Ausreißer-Verkauf an, den man gesondert betrachten muss.

*Erwartete Elemente:* die vier Werte vorher/nachher; der Gegensatz zwischen empfindlichen (Mittelwert, Standardabweichung) und robusten Kennzahlen (Median, Interquartilsabstand); eine differenzierte Schlussfolgerung.

## Aufgabe 1.6 — Herausforderung: zwei Geraden, derselbe mittlere Fehler

*Daten D0.1. Begriffe: vorzeichenbehaftete Abweichung und Betrag (Einleitung, Lektion 1.1), Standardabweichung (Lektion 1.1).* Nach Augenmaß gezeichnete Gerade: vorhergesagter Preis = 3 × Fläche + 30. Faule Gerade: vorhergesagter Preis = 222 k€ für alle Wohnungen.

In [ ]:
price = homes['price'].to_numpy(dtype=float)
floor_area = homes['floor_area'].to_numpy(dtype=float)

eye_line_errors = price - (3 * floor_area + 30)
lazy_line_errors = price - 222

In [ ]:
# Die drei Kennzahlen aus Frage b) (Standardabweichung: Division durch n)
if eye_line_errors is None or lazy_line_errors is None:
    print('Bitte füllen Sie die vorherige Zelle aus.')
else:
    indicators = pd.DataFrame({
        name: {'Mittel vorzeichenbeh. Abweichungen': e.mean(),
              'Mittel der Abweichungsbeträge': np.abs(e).mean(),
              'Std.-abw. vorzeichenbeh. Abweich.': e.std(ddof=0)}
        for name, e in [('Gerade nach Augenmaß', eye_line_errors), ('faule Gerade', lazy_line_errors)]
    })
    print('Abweichungen, Gerade nach Augenmaß:', eye_line_errors)
    print('Abweichungen, faule Gerade        :', lazy_line_errors)
    print('Prüfung:', 'richtig' if np.allclose(indicators.iloc[:, 0], [1/3, 44/6, 7.5645], atol=1e-3)
          and np.allclose(indicators.iloc[:, 1], [1/3, 64, 72.1472], atol=1e-3) else 'bitte prüfen')
    display(indicators.round(2))

In [ ]:
if eye_line_errors is not None and lazy_line_errors is not None:
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.axhline(0, color='black', linewidth=1)
    ax.scatter(floor_area - 0.8, eye_line_errors, color=ORANGE, label="Gerade nach Augenmaß")
    ax.scatter(floor_area + 0.8, lazy_line_errors, color=BLUE, marker='s', label='faule Gerade')
    ax.set_xlabel('Wohnfläche (m²)')
    ax.set_ylabel('Abweichung beobachtet − vorhergesagt (k€)')
    ax.legend()
    ax.grid(alpha=0.4)
    plt.show()

### Lösung zu Aufgabe 1.6

**a)** Abweichungen der faulen Geraden (beobachtet − 222): −104, −50, −37, +23, +57, +113 k€. Ihre Summe beträgt 2 und ihr Mittelwert ≈ **0,33 k€**. Das ist genau der Mittelwert der vorzeichenbehafteten Abweichungen der nach Augenmaß gezeichneten Geraden (−8, +7, −7, +5, −6, +11: Summe 2, Mittelwert ≈ 0,33 k€).

**b)**

| | Gerade nach Augenmaß | Faule Gerade |
|:--|--:|--:|
| Mittelwert der vorzeichenbehafteten Abweichungen | 0,33 k€ | 0,33 k€ |
| Mittelwert der Abweichungsbeträge | 44/6 ≈ 7,33 k€ | 384/6 = 64 k€ |
| Standardabweichung der vorzeichenbehafteten Abweichungen (Division durch $n$) | ≈ 7,56 k€ | ≈ 72,15 k€ |

Detail für die Gerade nach Augenmaß: Der Mittelwert beträgt 1/3. Die Abweichungen von diesem Mittelwert haben Quadrate mit der Summe 344 − 6 × (1/3)² ≈ 343,33. Die Varianz beträgt 343,33/6 ≈ 57,22 und die Standardabweichung ≈ 7,56 k€.

**c)** Die nach Augenmaß gezeichnete Gerade ist viel besser: Ihre Fehler betragen im Betrag etwa 7 k€, gegenüber 64 k€ bei der faulen Geraden. Der Mittelwert der Abweichungsbeträge und die Standardabweichung der vorzeichenbehafteten Abweichungen unterscheiden sie. Der Mittelwert der vorzeichenbehafteten Abweichungen schafft das nicht: Keine der beiden Geraden neigt zu einer Seite, und die Vorzeichen heben sich in beiden Fällen auf.

**d)** Die Abweichungen der faulen Geraden betragen $y_i - 222$. Das Abziehen derselben Konstanten verschiebt die ganze Reihe, ohne den Abstand jedes Werts zum Mittelwert zu ändern. Auch der Mittelwert sinkt um 222, also bleibt jede Abweichung vom Mittelwert $(y_i - 222) - (\bar{y} - 222) = y_i - \bar{y}$ gleich. Die Standardabweichung der Abweichungen ist daher die der Preise, ≈ 72,15 k€. Eine Gerade, die die Fläche ignoriert, irrt sich im Mittel um die Streuung der Preise selbst. Kapitel 3 wird diesen Vergleich nutzen.

*Erwartete Elemente:* die beiden gleichen Mittelwerte; zwei deutlich verschiedene Kennzahlen für die Fehlergröße; die Erklärung durch das Aufheben der Vorzeichen; das Verschiebungsargument für d).

In [ ]:
# Kontrolle zu d): Die Standardabweichung der Abweichungen der faulen Geraden ist gleich der der Preise
if lazy_line_errors is not None:
    print(round(lazy_line_errors.std(ddof=0), 4), round(price.std(ddof=0), 4))